# FIRMS Data Ingest
This notebook provides a clean, reusable FIRMS ingestion utility.
Usage: provide a `start_date` (YYYY-MM-DD), `days` (int), and `MAP_KEY` (your FIRMS/API key).
It will attempt to download data for common FIRMS satellites (MODIS, VIIRS_SNPP, VIIRS_NOAA20) and save a single combined CSV under `data/raw/`.

Note: `base_url` may need adjustment depending on the FIRMS API access method; update it if your instructor provided a different endpoint.

## Boundary Reference

Copy one of these boxes into your FIRMS filtering code. Coordinates are `[west, south, east, north]` in `EPSG:4326`.

If you do not pass a boundary box, the notebook fetches the whole world.
If you do pass a boundary box, use it like `boundary_box=[west, south, east, north]`.

Saved file names are kept simple:
- `modis_sp_20221009_3d_turkey.csv`
- `all_20221009_3d_world.csv`
- `viirs_snpp_sp_20221009_3d_uae.csv`

**Countries**
- Turkey: `[25.7, 35.8, 45.1, 42.2]`
- Ukraine: `[22.0, 44.0, 40.0, 52.5]`
- Iran: `[44.0, 25.0, 63.5, 39.9]`
- Qatar: `[50.6, 24.4, 51.7, 26.2]`
- United Arab Emirates: `[51.5, 22.5, 56.6, 26.5]`
- Saudi Arabia: `[34.4, 16.3, 55.7, 32.2]`
- Yemen: `[42.5, 12.0, 54.6, 19.0]`
- Israel: `[34.2, 29.4, 35.9, 33.4]`
- Lebanon: `[35.1, 33.0, 36.7, 34.7]`
- Syria: `[35.7, 32.0, 42.5, 37.4]`
- Iraq: `[38.8, 29.0, 49.5, 37.4]`
- Jordan: `[34.9, 29.0, 39.4, 33.5]`
- Kuwait: `[46.5, 28.4, 48.5, 30.1]`
- Bahrain: `[50.3, 25.3, 50.9, 26.5]`
- Oman: `[51.9, 16.5, 59.9, 26.5]`
- Australia: `[112.0, -44.0, 154.0, -10.0]`

**Cities**
- Bursa: `[28.45, 40.00, 30.00, 40.55]`
- Antalya: `[29.80, 36.55, 31.25, 37.15]`
- Izmir: `[26.95, 38.10, 27.65, 38.75]`

**Regions**
- Western Russia: `[27.0, 41.0, 61.0, 56.0]`
- Black Sea Basin: `[27.0, 40.0, 46.5, 47.5]`
- Eastern Mediterranean / Levant: `[33.0, 28.0, 43.5, 37.8]`
- Red Sea Region: `[32.0, 12.0, 44.0, 28.5]`
- Bab el-Mandeb: `[42.0, 11.0, 44.8, 14.8]`
- Strait of Hormuz: `[54.0, 24.0, 57.7, 27.5]`
- Persian Gulf: `[47.0, 23.0, 57.8, 31.8]`
- Texas Permian Basin: `[-104.5, 29.0, -100.5, 34.0]`

If you want tighter boxes later, I can narrow them to the exact city or corridor.

In [ ]:
import io
import time
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import requests


KNOWN_BOUNDARY_NAMES = {
    (25.7, 35.8, 45.1, 42.2): 'turkey',
    (22.0, 44.0, 40.0, 52.5): 'ukraine',
    (44.0, 25.0, 63.5, 39.9): 'iran',
    (50.6, 24.4, 51.7, 26.2): 'qatar',
    (51.5, 22.5, 56.6, 26.5): 'uae',
    (34.4, 16.3, 55.7, 32.2): 'saudi_arabia',
    (42.5, 12.0, 54.6, 19.0): 'yemen',
    (34.2, 29.4, 35.9, 33.4): 'israel',
    (35.1, 33.0, 36.7, 34.7): 'lebanon',
    (35.7, 32.0, 42.5, 37.4): 'syria',
    (38.8, 29.0, 49.5, 37.4): 'iraq',
    (34.9, 29.0, 39.4, 33.5): 'jordan',
    (46.5, 28.4, 48.5, 30.1): 'kuwait',
    (50.3, 25.3, 50.9, 26.5): 'bahrain',
    (51.9, 16.5, 59.9, 26.5): 'oman',
    (112.0, -44.0, 154.0, -10.0): 'australia',
    (28.45, 40.0, 30.0, 40.55): 'bursa',
    (29.8, 36.55, 31.25, 37.15): 'antalya',
    (26.95, 38.1, 27.65, 38.75): 'izmir',
    (27.0, 41.0, 61.0, 56.0): 'western_russia',
    (27.0, 40.0, 46.5, 47.5): 'black_sea_basin',
    (33.0, 28.0, 43.5, 37.8): 'levant',
    (32.0, 12.0, 44.0, 28.5): 'red_sea_region',
    (42.0, 11.0, 44.8, 14.8): 'bab_el_mandeb',
    (54.0, 24.0, 57.7, 27.5): 'strait_of_hormuz',
    (47.0, 23.0, 57.8, 31.8): 'persian_gulf',
    (-104.5, 29.0, -100.5, 34.0): 'texas_permian_basin',
    (34.0, 22.0, 64.0, 40.0): 'middleeastwar',
}


def _build_acq_timestamp(frame):
    if 'acq_date' not in frame.columns:
        return frame

    acq_date = pd.to_datetime(frame['acq_date'], errors='coerce').dt.strftime('%Y-%m-%d')
    if 'acq_time' in frame.columns:
        acq_time = frame['acq_time'].astype(str).str.replace('.0', '', regex=False).str.zfill(4).str[:4]
        frame['acq_timestamp'] = pd.to_datetime(acq_date + ' ' + acq_time, errors='coerce', format='%Y-%m-%d %H%M')
    else:
        frame['acq_timestamp'] = pd.to_datetime(acq_date, errors='coerce')
    return frame


def _slugify(value):
    text = str(value).strip().lower()
    for character in (' ', '/', '\\'):
        text = text.replace(character, '_')
    return text


def _box_key(boundary_box):
    return tuple(round(float(value), 4) for value in boundary_box)


def _format_boundary_box(boundary_box):
    if boundary_box is None:
        return 'world', 'world'

    if isinstance(boundary_box, str):
        boundary_value = boundary_box.strip()
        if not boundary_value or boundary_value.lower() == 'world':
            return 'world', 'world'
        return boundary_value, _slugify(boundary_value)

    if len(boundary_box) != 4:
        raise ValueError('boundary_box must contain four values in [west, south, east, north] order')

    west, south, east, north = boundary_box
    area_value = f"{west},{south},{east},{north}"
    area_slug = KNOWN_BOUNDARY_NAMES.get(_box_key(boundary_box))
    if area_slug is None:
        area_slug = 'bbox_' + '_'.join(str(value).replace('-', 'm').replace('.', 'p') for value in boundary_box)
    return area_value, area_slug


def _build_output_filename(source_name, start_dt, total_days, area_slug):
    return f"{source_name}_{start_dt.strftime('%Y%m%d')}_{int(total_days)}d_{area_slug}.csv"


def get_global_firms_data(start_date_str, total_days, map_key, boundary_box=None, boundary_label=None):
    """Fetch FIRMS fire/thermal data and save one combined CSV under data/raw/."""
    start_dt = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_dt = start_dt + timedelta(days=max(int(total_days) - 1, 0))
    days_ago = (datetime.now() - start_dt).days

    if days_ago > 10:
        satellites = ['MODIS_SP', 'VIIRS_SNPP_SP', 'VIIRS_NOAA20_SP']
        print('System: Historical date detected. Using Standard Processing (_SP) streams.')
    else:
        satellites = ['MODIS_NRT', 'VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT']
        print('System: Recent date detected. Using Near Real-Time (_NRT) streams.')

    area_value, area_slug = _format_boundary_box(boundary_box)
    if boundary_label:
        area_slug = _slugify(boundary_label)
    elif area_slug == 'world':
        area_slug = 'world'

    chunks = []
    current_date = start_dt
    remaining_days = total_days

    while remaining_days > 0:
        run_days = min(remaining_days, 5)
        chunks.append({'date': current_date.strftime('%Y-%m-%d'), 'range': run_days})
        current_date += timedelta(days=run_days)
        remaining_days -= run_days

    all_dataframes = []

    for sat in satellites:
        print(f"\nLaunching stream sequence for satellite sensor: {sat}")

        for chunk in chunks:
            target_date = chunk['date']
            day_range = chunk['range']

            print(f"  -> Extracting area footprint | Area: {area_slug} | Start: {target_date} | Duration: {day_range} days")

            url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sat}/{area_value}/{day_range}/{target_date}"

            try:
                response = requests.get(url, timeout=30)

                if response.status_code == 429:
                    print('  Rate limited by NASA servers. Waiting 10 seconds before retry...')
                    time.sleep(10)
                    response = requests.get(url, timeout=30)

                if response.status_code == 200:
                    if 'Invalid MAP_KEY' in response.text:
                        print('Critical Error: The MAP_KEY provided is invalid.')
                        return None

                    df_chunk = pd.read_csv(io.StringIO(response.text))

                    if not df_chunk.empty:
                        df_chunk = _build_acq_timestamp(df_chunk)
                        df_chunk['retrieved_at'] = datetime.utcnow()
                        df_chunk['satellite'] = sat
                        df_chunk['period_start'] = start_dt.strftime('%Y-%m-%d')
                        df_chunk['period_end'] = end_dt.strftime('%Y-%m-%d')
                        df_chunk['area_label'] = area_slug
                        all_dataframes.append(df_chunk)
                    else:
                        print('     No thermal observations registered in this area window.')
                else:
                    print(f"  Fetch failed for this segment. HTTP Status: {response.status_code}")

            except Exception as e:
                print(f"  Connection issue on endpoint: {e}")

            time.sleep(1.5)

    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)

        raw_dir = Path('data') / 'raw'
        raw_dir.mkdir(parents=True, exist_ok=True)
        combined_file = raw_dir / _build_output_filename('all', start_dt, total_days, area_slug)
        master_df.to_csv(combined_file, index=False)

        print('\n=========================================')
        print('PIPELINE EXECUTION SUCCESSFUL')
        print(f'Total Combined Area Records: {len(master_df)}')
        print(f'Saved combined CSV to: {combined_file}')
        print('=========================================')
        return master_df
    else:
        print('\nExtraction cycle complete: No records recovered.')
        return None

In [8]:
# --- Execution Parameters ---
MY_KEY = "4fe66d8195ccc519498954a7f8657d70"
START = "2025-06-01"  # Target starting baseline point (YYYY-MM-DD)
DURATION = 90  # Total days you want to fetch

# Optional: set a boundary box in [west, south, east, north] order.
# Leave it as None to fetch the whole world.
BOUNDARY_BOX = [25.7, 35.8, 45.1, 42.2]
# Example:
# BOUNDARY_BOX = [51.5, 22.5, 56.6, 26.5]  # UAE

# Run the function
df_world = get_global_firms_data(
    start_date_str=START,
    total_days=DURATION,
    map_key=MY_KEY,
    boundary_box=BOUNDARY_BOX,
)

# Inspect the dataset head to verify it's working
if df_world is not None:
    print(df_world.head())

System: Historical date detected. Using Standard Processing (_SP) streams.

Launching stream sequence for satellite sensor: MODIS_SP
  -> Extracting area footprint | Area: turkey | Start: 2025-06-01 | Duration: 5 days


C:\Users\AFT\AppData\Local\Temp\ipykernel_33088\2648161424.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_chunk['retrieved_at'] = datetime.utcnow()


  -> Extracting area footprint | Area: turkey | Start: 2025-06-06 | Duration: 5 days


C:\Users\AFT\AppData\Local\Temp\ipykernel_33088\2648161424.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_chunk['retrieved_at'] = datetime.utcnow()


  -> Extracting area footprint | Area: turkey | Start: 2025-06-11 | Duration: 5 days


...

  -> Extracting area footprint | Area: turkey | Start: 2025-08-20 | Duration: 5 days


C:\Users\AFT\AppData\Local\Temp\ipykernel_33088\2648161424.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_chunk['retrieved_at'] = datetime.utcnow()


  -> Extracting area footprint | Area: turkey | Start: 2025-08-25 | Duration: 5 days


C:\Users\AFT\AppData\Local\Temp\ipykernel_33088\2648161424.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_chunk['retrieved_at'] = datetime.utcnow()



PIPELINE EXECUTION SUCCESSFUL
Total Combined Area Records: 50653
Saved combined CSV to: data\raw\all_20250601_90d_turkey.csv
   latitude  longitude  brightness  scan  track    acq_date  acq_time  \
0   41.6367    42.9970       308.7   1.0    1.0  2025-06-01       653   
1   37.6281    45.0434       319.6   1.1    1.0  2025-06-01       654   
2   37.6299    45.0317       319.4   1.1    1.0  2025-06-01       654   
3   37.7379    45.0532       321.0   1.1    1.0  2025-06-01       654   
4   38.0453    40.3811       326.9   1.2    1.1  2025-06-01       654   

  satellite instrument confidence  ...   frp  daynight  type  \
0  MODIS_SP      MODIS         61  ...   5.5         D     2   
1  MODIS_SP      MODIS         33  ...   7.0         D     0   
2  MODIS_SP      MODIS         44  ...   7.4         D     2   
3  MODIS_SP      MODIS         36  ...   7.4         D     0   
4  MODIS_SP      MODIS         71  ...  15.5         D     0   

        acq_timestamp               retrieved_at p